## Hyperparameter Tuning

This notebook demonstrates hyperparameter tuning for a Multi-Layer Perceptron (MLP) classifier using the Breast Cancer Wisconsin dataset.

### What is Hyperparameter Tuning?

*   **Hyperparameters** are configuration settings for an MLP that are set before training (e.g., number of neurons, learning rate, batch size).
*   **Goal**: Find the best combination of hyperparameters that maximize accuracy and minimize loss.

### Two Common Tuning Methods

*   **Grid Search** - Tests all possible combinations of hyperparameters (exhaustive search).
*   **Random Search** - Randomly selects hyperparameter combinations (faster but less exhaustive).

### Step 1: Install Dependencies

In [ ]:
!pip install tensorflow scikit-learn

### Step 2: Load Dataset & Preprocess Data

We will use the Breast Cancer Wisconsin dataset, which contains 30 numeric features used to classify malignant or benign tumors.

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.optimizers import Adam
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV
from tensorflow.keras.wrappers.scikit_learn import KerasClassifier
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Load the dataset
data = load_breast_cancer()
X, y = data.data, data.target # X (features), y (target)

# Split dataset into training and testing sets (80% train, 20% test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Standardize feature values (important for neural networks)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

### Step 3: Define MLP Model Function

Create a function that builds an MLP model with variable hyperparameters.

In [ ]:
def create_mlp(hidden_units=16, activation='relu', optimizer='adam'):
    model = Sequential([
        Dense(hidden_units, activation=activation, input_shape=(30,)), # Input Layer (30 features)
        Dense(hidden_units, activation=activation), # Hidden Layer
        Dense(1, activation='sigmoid') # Output Layer (Binary classification)
    ])
    model.compile(optimizer=optimizer, loss='binary_crossentropy', metrics=['accuracy'])
    return model

### Step 4: Train Baseline MLP (Without Tuning)

In [ ]:
# Train baseline model
baseline_model = create_mlp()

history = baseline_model.fit(X_train, y_train, epochs=10, batch_size=32, validation_data=(X_test, y_test), verbose=0)

# Evaluate the baseline model
baseline_loss, baseline_acc = baseline_model.evaluate(X_test, y_test, verbose=0)
print(f"\nBaseline Test Accuracy: {baseline_acc * 100:.2f}%")

### Step 5: Hyperparameter Tuning with Grid Search

Grid Search tests all possible hyperparameter combinations.

In [ ]:
# Wrap MLP model for Scikit-learn compatibility
model = KerasClassifier(build_fn=create_mlp, verbose=0)

# Define hyperparameter grid
param_grid = {
    'hidden_units': [8, 16, 32],
    'activation': ['relu', 'tanh'],
    'optimizer': ['adam', 'sgd'],
    'batch_size': [16, 32],
    'epochs': [10, 20]
}

# Perform Grid Search
grid_search = GridSearchCV(estimator=model, param_grid=param_grid, cv=3, n_jobs=-1)
grid_search.fit(X_train, y_train)

# Best parameters found
print("Best Hyperparameters:", grid_search.best_params_)

### Step 6: Hyperparameter Tuning with Random Search

Random Search randomly selects hyperparameters (faster, but not exhaustive).

In [ ]:
# Perform Random Search
random_search = RandomizedSearchCV(estimator=model, param_distributions=param_grid, n_iter=5, cv=3, n_jobs=-1)
random_search.fit(X_train, y_train)

# Best parameters found
print("Best Hyperparameters (Random Search):", random_search.best_params_)

### Step 7: Train MLP with Optimized Hyperparameters

Train a new model using the best parameters found by Grid Search.

In [ ]:
# Train model with best Grid Search parameters
best_params = grid_search.best_params_
optimized_model = create_mlp(hidden_units=best_params['hidden_units'], activation=best_params['activation'], optimizer=best_params['optimizer'])

optimized_model.fit(X_train, y_train, epochs=best_params['epochs'], batch_size=best_params['batch_size'], validation_data=(X_test, y_test), verbose=0)

# Evaluate optimized model
optimized_loss, optimized_acc = optimized_model.evaluate(X_test, y_test, verbose=0)
print(f"\nOptimized Test Accuracy: {optimized_acc * 100:.2f}%")

### Step 8: Compare Results

**RESULT:**

Thus, the implementation of the Multi-Layer Perceptron (MLP) classifier for the Breast Cancer dataset was successfully completed, including hyperparameter tuning using Grid Search. The results before and after optimization were successfully compared.

## Resources